In [1]:
# Imports
import boto3
from sagemaker import get_execution_role
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
role = get_execution_role()
bucket = 'json-avalos-ads508'
delog_data_key = 'Delivery_Logistics 2.csv'
dysulog_data_key = 'dynamic_supply_chain_logistics_dataset.csv'
trade_cus_data_key = 'trade_customs_dataset 2.csv'
logistics_data_key = 'Logistics_supply_chain.csv'
del_loc = 's3://{}/{}'.format(bucket, delog_data_key)
dyn_loc = 's3://{}/{}'.format(bucket, dysulog_data_key)
trade_loc = 's3://{}/{}'.format(bucket, trade_cus_data_key)
logistics_loc = 's3://{}/{}'.format(bucket, logistics_data_key)

In [3]:
!pip install openpyxl # dependency
delivery = pd.read_csv(del_loc)
dynamic = pd.read_csv(dyn_loc)
trade = pd.read_csv(trade_loc)
logistics = pd.read.csv(logistics_loc)

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)


In [ ]:
def clean_trade(df: pd.DataFrame) -> pd.DataFrame:
    """trade_customs_dataset — parse dates, derive delay columns, basic QA."""
    df = df.copy()
    # Parse datetime columns
    for col in ["Shipment_Date", "Estimated_Arrival_Date", "Actual_Arrival_Date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    # Drop rows where we can't compute the delay
    before = len(df)
    df = df.dropna(subset=["Estimated_Arrival_Date", "Actual_Arrival_Date"])
    print(f"  trade: dropped {before - len(df)} rows with missing arrival dates")
    # Derived delay features
    df["Transit_Delay_Days"] = (
        df["Actual_Arrival_Date"] - df["Estimated_Arrival_Date"]
    ).dt.days
    df["Is_Delayed"] = (df["Transit_Delay_Days"] > 0).astype(int)
    # Total delay = transit + customs
    if "Customs_Delay_Days" in df.columns:
        df["Total_Delay_Days"] = df["Transit_Delay_Days"] + df["Customs_Delay_Days"].fillna(0)
    # Temporal features
    df["Shipment_Month"] = df["Shipment_Date"].dt.month
    df["Shipment_DayOfWeek"] = df["Shipment_Date"].dt.dayofweek   # 0=Mon
    df["Shipment_IsWeekend"] = (df["Shipment_DayOfWeek"] >= 5).astype(int)
    # Normalise carrier / route strings
    for col in ["Carrier_Name", "Route_Code"]:
        if col in df.columns:
            df[col] = df[col].str.strip().str.upper()
    return df


def clean_delivery(df: pd.DataFrame) -> pd.DataFrame:
    """Delivery_Logistics — sanity checks, derived efficiency features."""
    df = df.copy()
    # Guard against zero denominators
    df = df[df["distance_km"] > 0].copy()
    df = df[df["expected_time_hours"] > 0].copy()
    # Derived features
    df["time_vs_expected"] = df["delivery_time_hours"] / df["expected_time_hours"]
    df["cost_per_km"] = df["delivery_cost"] / df["distance_km"]
    # Standardise the delay flag name (some versions use 'delayed', others 'Delayed')
    delay_col = next((c for c in df.columns if c.lower() == "delayed"), None)
    if delay_col and delay_col != "delayed":
        df = df.rename(columns={delay_col: "delayed"})
    return df


def clean_dynamic(df: pd.DataFrame) -> pd.DataFrame:
    """dynamic_supply_chain_logistics_dataset — cast types, derive features."""
    df = df.copy()
    # Ensure congestion columns are numeric integers
    for col in ["port_congestion_level", "traffic_congestion_level"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    # Combined congestion index
    df["combined_congestion"] = (
        df.get("port_congestion_level", 0) + df.get("traffic_congestion_level", 0)
    )
    # Risk × clearance interaction
    if "route_risk_level" in df.columns and "customs_clearance_time" in df.columns:
        df["risk_clearance_product"] = (
            df["route_risk_level"] * df["customs_clearance_time"]
        )
    # Log-transform shipping costs (right-skewed)
    if "shipping_costs" in df.columns:
        df["log_shipping_costs"] = np.log1p(df["shipping_costs"])
    return df


def _iqr_bounds(series: pd.Series, k: float = 3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def clean_logistics(df: pd.DataFrame) -> pd.DataFrame:
    """Logistics_supply_chain — timestamp parse, GPS outlier removal, interactions."""
    df = df.copy()
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    # Remove GPS outliers (keep only plausible lat/lon ranges)
    for lat_col in [c for c in df.columns if "latitude" in c.lower()]:
        lo, hi = _iqr_bounds(df[lat_col])
        before = len(df)
        df = df[(df[lat_col] >= lo) & (df[lat_col] <= hi)]
        print(f"  logistics: GPS lat outlier filter removed {before - len(df)} rows")
    for lon_col in [c for c in df.columns if "longitude" in c.lower()]:
        lo, hi = _iqr_bounds(df[lon_col])
        before = len(df)
        df = df[(df[lon_col] >= lo) & (df[lon_col] <= hi)]
        print(f"  logistics: GPS lon outlier filter removed {before - len(df)} rows")
    # Geographic grid bins (0.5° resolution)
    for lat_col in [c for c in df.columns if "latitude" in c.lower()]:
        df["lat_bin"] = (df[lat_col] / 0.5).round(0) * 0.5
    for lon_col in [c for c in df.columns if "longitude" in c.lower()]:
        df["lon_bin"] = (df[lon_col] / 0.5).round(0) * 0.5
    # Weather × congestion interaction
    if "weather_condition_severity" in df.columns and "traffic_congestion_level" in df.columns:
        df["weather_congestion_interaction"] = (
            df["weather_condition_severity"] * df["traffic_congestion_level"]
        )
    # Supplier reliability tier (low / medium / high)
    if "supplier_reliability_score" in df.columns:
        df["supplier_tier"] = pd.cut(
            df["supplier_reliability_score"],
            bins=[0, 0.4, 0.7, 1.0],
            labels=["low", "medium", "high"],
            include_lowest=True,
        )
    return df

In [ ]:
trade = clean_trade(trade)
delivery = clean_delivery(delivery)
dynamic = clean_dynamic(dynamic)
logistics = clean_logistics(logistics)